In [51]:
from scipy.signal import lfilter
import numpy as np
import librosa

In [52]:
# Load vocal source (for LPC analysis)
voice, sr = librosa.load("voice.wav", sr=None)

# Load instrument / material to be filtered
instrument, sr2 = librosa.load("instrument.wav", sr=None)

In [53]:
# Ensure same sample rate
assert sr == sr2

In [54]:
frame_length = int(0.03 * sr)
hop_length = frame_length // 2
order = 16

In [55]:
voice = np.pad(voice, (0, frame_length))
instrument = np.pad(instrument, (0, frame_length))

In [56]:
voice_frames = librosa.util.frame(voice, frame_length=frame_length, hop_length=hop_length)
inst_frames = librosa.util.frame(instrument, frame_length=frame_length, hop_length=hop_length)

In [57]:
output = np.zeros(len(instrument))
window = np.hamming(frame_length)
norm = np.zeros(len(instrument))

In [63]:
for i in range(min(voice_frames.shape[1], inst_frames.shape[1])):

    # --- LPC analysis on VOICE ---
    v_frame = voice_frames[:, i] * window
    a = librosa.lpc(v_frame, order=10)

    if i > 0:
        a = 0.85 * prev_a + 0.15 * a
    
    prev_a = a

    # --- INSTRUMENT excitation ---
    x_frame = inst_frames[:, i] * window
    x_frame = x_frame / (np.std(x_frame) + 1e-8)

    # treat instrument as excitation
    residual = x_frame

    # apply vocal filter to instrument
    filtered = lfilter([1.0], a, residual)
    shaped = 0.6 * residual + 0.4 * filtered    

    # overlap-add reconstruction
    start = i * hop_length
    output[start:start + frame_length] += shaped * window
    norm[start:start + frame_length] += window**2

In [64]:
output /= (norm + 1e-8)

import soundfile as sf
sf.write("instrument_through_voice.wav", output, sr)